### Create DB if file does not exists and add data_import_control Table

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
import os
from datetime import datetime
import re
from datetime import datetime


# config

photometer_name = 'stars928'
db_name = f'../data/tess_data.db'
ecsv_folder = f'../python/TESS-IDA-TOOLS/jupyter/ECSV/{photometer_name}/'

# SQLite-Verbindung erstellen
engine = create_engine(f'sqlite:///{db_name}')

# Tabelle in SQLite erstellen (falls nicht vorhanden)
def execute_query(query):
    with engine.connect() as connection:
        result = connection.execute(text(query))
        return result.fetchall()

def execute_command(command):
    with engine.connect() as connection:
        connection.execute(text(command))
        connection.commit()


# Example usage:
command = """
CREATE TABLE IF NOT EXISTS data_import_control (
    name TEXT NOT NULL,
    date_of_data_name TIMESTAMP NOT NULL,
    date_of_import TIMESTAMP NOT NULL,
    complete BOOLEAN NOT NULL
);
"""

execute_command(command)




### Create Photometer Table if it does not exists 

In [8]:
query = f"""SELECT EXISTS (SELECT 1 FROM sqlite_master WHERE type='table' AND name='{photometer_name}_data') AS table_exists;"""
result = execute_query(query)
print

if result == [(0,)]:
    create_table_photometer_query = f'''
    CREATE TABLE {photometer_name}_data (
        time TIMESTAMP,
        enclosure_temperature DOUBLE PRECISION,
        sky_temperature DOUBLE PRECISION,
        frequency DOUBLE PRECISION,
        msas DOUBLE PRECISION,
        zp DOUBLE PRECISION,
        sequence_number BIGINT,
        sun_alt DOUBLE PRECISION,
        moon_alt DOUBLE PRECISION,
        moon_illumination DOUBLE PRECISION
    );
    '''

    # create table
    with engine.connect() as connection:
        connection.execute(text(create_table_photometer_query))
    print("created table")
else:
    print("exists")

exists


### Photometer Table

In [3]:
command = f"DROP TABLE IF EXISTS {photometer_name}_data"
execute_command(command)


### Create Photometer Table

In [4]:
create_table_photometer_query = f'''
CREATE TABLE {photometer_name}_data (
    time TIMESTAMP,
    enclosure_temperature DOUBLE PRECISION,
    sky_temperature DOUBLE PRECISION,
    frequency DOUBLE PRECISION,
    msas DOUBLE PRECISION,
    zp DOUBLE PRECISION,
    sequence_number BIGINT,
    sun_alt DOUBLE PRECISION,
    moon_alt DOUBLE PRECISION,
    moon_illumination DOUBLE PRECISION
);
'''

# create table
with engine.connect() as connection:
    connection.execute(text(create_table_photometer_query))

### Create entry in the data_import_control for the file

In [111]:
import datetime

# Set the data variable
data = "stars928_2024-08"
data = data.split("_")[1].replace("-", "")  # Extract date portion and remove the hyphen

# Convert the extracted data to a datetime object and set to first day of the month
date_of_data_name = datetime.datetime.strptime(data, "%Y%m").date().replace(day=1)  # Correct format for date_of_data_name

# Current datetime for date_of_import
date_of_import = datetime.datetime.now().date()

# Calculate the first day of the next month after date_of_data_name
next_month = (date_of_data_name.replace(day=1) + datetime.timedelta(days=31)).replace(day=1)

# Insert command with updated conditional `complete` value
command = f"""
INSERT INTO data_import_control (name, date_of_data_name, date_of_import, complete) 
VALUES (
    'stars928',
    '{date_of_data_name}',
    '{date_of_import}',
    CASE WHEN date('{date_of_import}') >= date('{next_month}') THEN 1 ELSE 0 END
);
"""
# Execute the command
execute_command(command)


### Test if there need to get new data

In [3]:
# Generate a list of months between start and end dates
from datetime import datetime, timedelta

start_date = datetime.strptime("2024-01-01", "%Y-%m-%d")
end_date = datetime.strptime("2024-09-04", "%Y-%m-%d")
date_list = []


# Populate date_list with the first of each month in the range
current_date = start_date
while current_date <= end_date:
    date_list.append(current_date.strftime("%Y-%m-%d"))
    # Move to the first of the next month
    next_month = current_date.month % 12 + 1
    next_year = current_date.year + (current_date.month // 12)
    current_date = current_date.replace(year=next_year, month=next_month, day=1)

# Fetch existing dates in the database
photometer_name = "stars928"
query = f"""
SELECT date_of_data_name
FROM data_import_control
WHERE name = '{photometer_name}'
AND complete = 1
AND date_of_data_name BETWEEN '{start_date}' AND '{end_date}'
"""

# Execute the query and get results
existing_dates = execute_query(query)

# Extract dates from the query result as a set
existing_dates = {row[0] for row in existing_dates}

# Remove existing dates from date_list
filtered_dates = [date for date in date_list if date not in existing_dates]

# Print filtered dates
required_month_files = [date[:7] for date in filtered_dates]  # Extracts "YYYY-MM" from each date
print(required_month_files)



['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07']


### Download required files from list

So i need to install the TESS-IDA lib

In [ ]:
import os

# List of dates
dates = ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07']

# Iterate through each date and run the command
for date in dates:
    command = f"tess-ida-pipe --console single --in-dir IDA --out-dir ECSV --name stars289 --month {date}"
    print(f"Running command: {command}")
    os.system(command)  # Executes the command in the shell


In [ ]:
# # Muster zur Überprüfung des Dateinamensformats
filename_pattern = re.compile(rf"{photometer_name}_\d{{4}}-\d{{2}}\.ecsv")

# Importieren und Filtern der ECSV-Daten
for file_name in os.listdir(ecsv_folder):
    # Überprüfen, ob der Dateiname auf ".ecsv" endet und dem Format entspricht
    if file_name.endswith('.ecsv') and filename_pattern.match(file_name):
        ecsv_file = os.path.join(ecsv_folder, file_name)
        data = pd.read_csv(ecsv_file, comment='#', delimiter=',')

        # Spaltennamen anpassen
        data.columns = [
            'time', 
            'enclosure_temperature', 
            'sky_temperature', 
            'frequency', 
            'msas', 
            'zp', 
            'sequence_number', 
            'sun_alt', 
            'moon_alt', 
            'moon_illumination'
        ]

        # Gefilterte Daten in die SQLite-Datenbank importieren
        data.to_sql(f'{photometer_name}_data', engine, if_exists='append', index=False)
        print(f"Gefilterte Daten von {file_name} wurden erfolgreich importiert.")
    else:
        print(f"Datei {file_name} entspricht nicht dem erwarteten Format und wird übersprungen.")

print("Alle gefilterten Dateien wurden in die SQLite-Datenbank importiert.")
